# Idiap / USSD teachers: 188 participant internal 80:10:10 protocol, seed 103

Run cells in order. This notebook starts with the **split and provenance audit**. It combines the 107 original TRAIN participants, 34 usable DEV participants (440 excluded), and 47 locally labeled original TEST participants into a new, stratified, participant-disjoint TRAIN-150 / VAL-19 / internal TEST-19 split. Internal TEST labels are used solely to stratify and freeze membership. Their features and predictions must remain unopened until the final evaluation.

**Checkpoint rule:** The published Idiap and USSD checkpoints already saw some participants who may now be in VAL or internal TEST. Do not reuse those weights for validation or final scoring. Fresh instances of the same architectures must be trained on TRAIN-150. This is an internal redistributed split, not an official AVEC test result. The provenance of local `full_test_split.csv` labels has not been independently verified. Earlier results on other splits cannot be compared as if this were an untouched test.

Stage 1 below saves only the split. We will add preprocessing/teacher stages to this same notebook after the split audit is confirmed. The older experiment folders are left untouched.


## 1. Mount Drive and load the pinned GitHub branch


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import base64, os, subprocess, sys
DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
assert (DAIC / 'metadata').is_dir(), f'Missing metadata under {DAIC}'
REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
BRANCH = 'feature/official141-original-teachers-80-10-10'
ROOT = Path('/content/reliability-aware-depression-kd')
token = userdata.get('GITHUB_TOKEN'); assert token, 'Add GITHUB_TOKEN to Colab Secrets'
header = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy(); env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {header}')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', str(ROOT)], check=True, env=env)
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=env)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
CODE = Path('/content/reliability-aware-depression-kd-original188-' + revision[:12])
if not CODE.exists(): subprocess.run(['git', '-C', str(ROOT), 'worktree', 'add', '--detach', str(CODE), revision], check=True)
assert (CODE / '.git').exists() and subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() == revision
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-colab.txt')], check=True)
del token, header, env
print('Revision:', revision)


## 2. Freeze the 150 / 19 / 19 split


In [ ]:
import pandas as pd
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
from scripts.experiments.internal188.split import main as split_stage
EXP = DAIC / 'experiments' / 'idiap_ussd_internal188_seed103'
split_stage(['--daic-root', str(DAIC), '--output', str(EXP / 'split'), '--seed', '103'])
manifest = pd.read_csv(EXP / 'split' / 'manifest.csv')
assert len(manifest) == 188 and manifest.participant_id.nunique() == 188
assert 440 not in set(manifest.participant_id)
assert manifest.split.value_counts().to_dict() == {'train': 150, 'val': 19, 'student_test': 19}
print(pd.crosstab(manifest.split, manifest.label))
print('Source audit:', manifest.groupby(['split', 'source_split']).size().to_dict())
print('Saved:', EXP / 'split' / 'manifest.csv')


## Stop after the split audit

Share the printed class-count and source tables. The next stage will validate transcripts and audio coverage for TRAIN and VAL only, then train and evaluate the text teacher, followed by the audio teacher. Internal TEST evaluation stays closed; no OOF.
